In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import textstat

df = pd.read_csv('../output/lada/legalbench_lada_3k.csv')

In [2]:
from collections import defaultdict

def cap_by_type(df_in, type_col='type', max_per_type=25):
    # preserve the original row order while limiting each type to max_per_type
    counts = defaultdict(int)
    keep_idx = []
    for idx, t in zip(df_in.index, df_in[type_col]):
        if counts[t] < max_per_type:
            keep_idx.append(idx)
            counts[t] += 1
    return df_in.loc[keep_idx]

# F1 analysis

In [3]:
top_f1 = df.sort_values('Factor_1', ascending=False).head(100)

top_f1 = cap_by_type(top_f1)

print(top_f1['type'].value_counts())

top_f2 = df.sort_values('Factor_2', ascending=False).head(100)

top_f2 = cap_by_type(top_f2)

print(top_f2['type'].value_counts())

top_f3 = df.sort_values('Factor_3', ascending=False).head(100)

top_f3 = cap_by_type(top_f3)

print(top_f3['type'].value_counts())

type
international_citizenship       25
function_of_decision_section     2
abercrombie                      1
Name: count, dtype: int64
type
corporate_lobbying    25
Name: count, dtype: int64
type
international_citizenship    25
corporate_lobbying            2
Name: count, dtype: int64


In [4]:
word_pattern = re.compile(r"\b\w+\b")
sentiment_analyzer = SentimentIntensityAnalyzer()

def compute_text_metrics(series, include_sentiment_label=False):
    metrics = []
    for text in series.fillna(""):
        text_str = str(text)
        tokens = word_pattern.findall(text_str.lower())
        token_lengths = [len(token) for token in tokens]
        question_length = len(tokens)
        avg_length = float(np.mean(token_lengths)) if token_lengths else 0.0
        burstiness = float(np.std(token_lengths) / avg_length) if token_lengths and avg_length else 0.0
        if token_lengths:
            counts = Counter(tokens)
            total = sum(counts.values())
            probs = np.array(list(counts.values()), dtype=float) / total
            entropy = float(-np.sum(probs * np.log(probs)))
            perplexity = float(np.exp(entropy))
        else:
            perplexity = 0.0
        flesch_kincaid = float(textstat.flesch_kincaid_grade(text_str)) if text_str.strip() else 0.0
        sentiment_scores = sentiment_analyzer.polarity_scores(text_str) if text_str.strip() else {"compound": 0.0}
        compound_sentiment = float(sentiment_scores.get("compound", 0.0))
        record = {
            "question": text_str,
            "question_length": question_length,
            "average_word_length": avg_length,
            "burstiness": burstiness,
            "perplexity": perplexity,
            "flesch_kincaid_grade": flesch_kincaid,
            "sentiment_compound": compound_sentiment
        }
        if include_sentiment_label:
            if compound_sentiment >= 0.05:
                sentiment_label = "positive"
            elif compound_sentiment <= -0.05:
                sentiment_label = "negative"
            else:
                sentiment_label = "neutral"
            record["sentiment_label"] = sentiment_label
        metrics.append(record)
    return pd.DataFrame(metrics)

In [5]:
df_metrics = compute_text_metrics(df['question'])
top_f1_metrics = compute_text_metrics(top_f1['question'])
top_f2_metrics = compute_text_metrics(top_f2['question'])
top_f3_metrics = compute_text_metrics(top_f3['question'])


summary_columns = ["question_length", "average_word_length", "burstiness", "perplexity", "flesch_kincaid_grade", "sentiment_compound"]
summary_df = pd.concat(
    [
        df_metrics[summary_columns].mean().rename("overall_mean"),
        top_f1_metrics[summary_columns].mean().rename("top_f1_mean"),
        top_f2_metrics[summary_columns].mean().rename("top_f2_mean"),
        top_f3_metrics[summary_columns].mean().rename("top_f3_mean")
    ],
    axis=1
)

display(summary_df)

for label, metrics_df in [
    ("overall", df_metrics),
    ("top_f1", top_f1_metrics),
    ("top_f2", top_f2_metrics),
    ("top_f3", top_f3_metrics)
]:
    print(f"\nSample metrics for {label} questions:")
    display(metrics_df.head(5))
    if "sentiment_label" in metrics_df.columns:
        print("Sentiment label distribution:")
        display(metrics_df['sentiment_label'].value_counts(normalize=True).rename(lambda x: f"{x} ({label})"))

,overall_mean,top_f1_mean,top_f2_mean,top_f3_mean
question_length,256.478217,34.892857,917.480000,90.407407
average_word_length,5.226676,5.151220,5.613108,5.092792
burstiness,0.568251,0.593159,0.553853,0.613801
perplexity,72.579784,25.677773,207.617018,36.938509
flesch_kincaid_grade,13.632242,11.533587,18.634092,13.571834
sentiment_compound,0.207861,-0.036236,0.993000,0.148533



Sample metrics for overall questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,"Description: The mark ""7-Eleven"" for a conveni...",17,4.352941,0.649493,15.668724,6.875000,0.0000
1,"Description: The mark ""Airbus"" for an airplane...",8,6.125000,0.585469,8.000000,11.130000,0.0000
2,"Description: The mark ""Amazon"" for an online s...",9,5.555556,0.488262,9.000000,8.897778,0.1779
3,"Description: The mark ""American Airlines"" for ...",11,6.090909,0.566378,11.000000,11.227273,0.0000
4,"Description: The mark ""Antilds"" for plant seeds.",7,5.428571,0.480939,7.000000,3.997143,0.0000



Sample metrics for top_f1 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Question: Consider the country of Mali. Does t...,29,5.172414,0.596583,22.834660,11.630517,-0.3182
1,Question: Consider the country of Turkmenistan...,29,5.448276,0.608912,22.834660,12.444310,-0.3182
2,Question: Consider the country of Mauritania. ...,30,5.200000,0.567009,24.505964,9.926667,0.6249
3,Question: Consider the country of Antigua and ...,31,5.258065,0.577600,23.704672,12.532419,-0.3182
4,Question: Consider the country of Lebanon. Doe...,41,5.097561,0.695675,29.480801,13.702561,-0.8442



Sample metrics for top_f2 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Official title of bill: To amend the Occupatio...,987,5.645390,0.536968,243.673455,20.224711,0.9927
1,Official title of bill: A bill to jump-start e...,943,5.858961,0.535488,211.092875,18.112843,0.9959
2,Official title of bill: To foster commercial r...,1024,5.831055,0.524980,241.344475,19.293670,0.9987
3,Official title of bill: A bill to require the ...,899,5.173526,0.542474,179.475628,17.770071,0.9969
4,Official title of bill: To amend the Internal ...,840,5.621429,0.566265,197.395701,17.731394,0.9904



Sample metrics for top_f3 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Question: Consider the country of United Arab ...,35,5.142857,0.550252,28.285691,13.149286,0.1280
1,Question: Consider the country of Cambodia. Do...,35,4.914286,0.574259,25.116646,10.452143,0.6249
2,Question: Consider the country of Antigua and ...,39,4.794872,0.616111,27.572374,12.589359,0.5106
3,Question: Consider the country of Guinea Bissa...,42,5.095238,0.686328,30.437943,13.671429,-0.8442
4,Question: Consider the country of Sudan. Does ...,53,5.094340,0.669500,31.462799,16.118585,-0.3182
